<div style="font-size: 13px; line-height: 1.35;">

<h3 style="font-size: 15px; margin: 8px 0 4px 0;">Entity &amp; Relationship Extraction — Simple Explanation</h3>

<p style="font-size: 13px; margin: 4px 0;"><strong>The Problem</strong><br>
Your RAG system stores chunks of prose. When you ask a bridge question like <em>"What do malaria and rice blast have in common?"</em>, the retriever grabs the 4 most similar chunks — often all about one topic, because similarity search doesn't understand relationships. The LLM sees 4 chunks about cassava (or all malaria, or all rice), never both, and correctly says: <em>"I don't know."</em></p>

<p style="font-size: 13px; margin: 4px 0;"><strong>The Idea</strong><br>
Instead of storing only paragraphs, we break each paragraph into tiny facts called <strong>triples</strong>:</p>

<pre style="font-size: 12px; line-height: 1.2; margin: 4px 0; padding: 4px;"><code>Subject  →  Relation  →  Object</code></pre>

<p style="font-size: 13px; margin: 4px 0;">Examples:</p>

<pre style="font-size: 12px; line-height: 1.2; margin: 4px 0; padding: 4px;"><code>Malaria        →  is a      →  Disease
Rice Blast     →  is a      →  Disease
Malaria        →  caused by →  Parasite
Cassava Mosaic →  affects   →  Cassava plants</code></pre>

<p style="font-size: 13px; margin: 4px 0;">Each triple is one small, comparable fact. When many triples share entities, they form a <strong>knowledge graph</strong> — a web of connected facts.</p>

<p style="font-size: 13px; margin: 4px 0;"><strong>Why Triples Help</strong><br>
When you ask <em>"What do malaria and rice blast have in common?"</em>, the system can:</p>

<ol style="font-size: 13px; margin: 4px 0; padding-left: 20px;">
  <li>Look at all triples with <code>Malaria</code> as the subject.</li>
  <li>Look at all triples with <code>Rice Blast</code> as the subject.</li>
  <li>Compare the two lists.</li>
  <li>Find shared relations: both are <code>is a Disease</code>, both are <code>caused by something</code>.</li>
  <li>Answer: <em>"Both are diseases, and both are caused by a pathogen."</em></li>
</ol>

<p style="font-size: 13px; margin: 4px 0;">This is impossible with plain chunks — but easy with triples.</p>

<p style="font-size: 13px; margin: 4px 0;"><strong>Where Triples Come From</strong><br>
Triples don't exist in the text. So we use an <strong>LLM</strong> to extract them.</p>

<p style="font-size: 13px; margin: 4px 0;">We give the LLM:</p>

<blockquote style="font-size: 12px; margin: 4px 0 4px 12px; padding-left: 8px; border-left: 3px solid #ccc;"><em>"Malaria is a mosquito-borne disease caused by Plasmodium parasites. It affects mostly children under five."</em></blockquote>

<p style="font-size: 13px; margin: 4px 0;">And instruct it to output:</p>

<pre style="font-size: 12px; line-height: 1.2; margin: 4px 0; padding: 4px;"><code>Malaria | is a           | Disease
Malaria | caused by      | Plasmodium Parasites
Malaria | transmitted by | Mosquitoes
Malaria | affects        | Children Under Five</code></pre>

<p style="font-size: 13px; margin: 4px 0;">That process is called <strong>entity and relationship extraction</strong>.</p>

<p style="font-size: 13px; margin: 4px 0;"><strong>The Extraction Process</strong></p>

<pre style="font-size: 12px; line-height: 1.2; margin: 4px 0; padding: 4px;"><code>Step 1: Take one chunk of text.
Step 2: Send it to the LLM with an extraction prompt.
Step 3: The LLM returns a list of triples.
Step 4: Repeat for all 80 chunks.
Step 5: Save all triples to a JSON file.</code></pre>

<p style="font-size: 13px; margin: 4px 0;">This notebook (<code>02_entity_extraction.ipynb</code>) does exactly those five steps.</p>

<p style="font-size: 13px; margin: 4px 0;"><strong>Two Rules That Keep the Graph Clean</strong></p>

<p style="font-size: 13px; margin: 4px 0;"><strong>Rule 1 — Canonical names.</strong> Every entity is written the same way every time.</p>
<ul style="font-size: 13px; margin: 4px 0; padding-left: 20px;">
  <li>✅ <code>Malaria</code>, <code>Cassava Mosaic Disease</code>, <code>Nigeria</code></li>
  <li>❌ <code>malaria</code>, <code>the disease malaria</code>, <code>CMD</code></li>
</ul>
<p style="font-size: 13px; margin: 4px 0;"><em>Why:</em> if the LLM writes "Malaria" and "malaria" as two different things, they become two separate nodes and never connect.</p>

<p style="font-size: 13px; margin: 4px 0;"><strong>Rule 2 — Controlled relation vocabulary.</strong> Relations come from a fixed list:</p>

<table style="font-size: 12px; margin: 4px 0; border-collapse: collapse;">
  <thead>
    <tr><th style="text-align: left; padding: 3px 8px; border-bottom: 1px solid #ccc;">Relation</th><th style="text-align: left; padding: 3px 8px; border-bottom: 1px solid #ccc;">Meaning</th></tr>
  </thead>
  <tbody>
    <tr><td style="padding: 3px 8px;"><code>IS_A</code></td><td style="padding: 3px 8px;">Category membership</td></tr>
    <tr><td style="padding: 3px 8px;"><code>CAUSED_BY</code></td><td style="padding: 3px 8px;">Pathogen or origin</td></tr>
    <tr><td style="padding: 3px 8px;"><code>TRANSMITTED_BY</code></td><td style="padding: 3px 8px;">Vector</td></tr>
    <tr><td style="padding: 3px 8px;"><code>AFFECTS</code></td><td style="padding: 3px 8px;">Target of impact</td></tr>
    <tr><td style="padding: 3px 8px;"><code>HAS_SYMPTOM</code></td><td style="padding: 3px 8px;">Symptom of a disease</td></tr>
    <tr><td style="padding: 3px 8px;"><code>CONTROL</code></td><td style="padding: 3px 8px;">Management method</td></tr>
  </tbody>
</table>

<p style="font-size: 13px; margin: 4px 0;"><strong>Summary</strong></p>

<table style="font-size: 12px; margin: 4px 0; border-collapse: collapse;">
  <thead>
    <tr><th style="text-align: left; padding: 3px 8px; border-bottom: 1px solid #ccc;">Term</th><th style="text-align: left; padding: 3px 8px; border-bottom: 1px solid #ccc;">Meaning</th></tr>
  </thead>
  <tbody>
    <tr><td style="padding: 3px 8px;">Triple</td><td style="padding: 3px 8px;">One atomic fact: subject, relation, object</td></tr>
    <tr><td style="padding: 3px 8px;">Entity</td><td style="padding: 3px 8px;">A thing — a node in the graph</td></tr>
    <tr><td style="padding: 3px 8px;">Relation</td><td style="padding: 3px 8px;">A connection between two entities</td></tr>
    <tr><td style="padding: 3px 8px;">Extraction</td><td style="padding: 3px 8px;">Converting prose into triples with an LLM</td></tr>
    <tr><td style="padding: 3px 8px;">Canonical name</td><td style="padding: 3px 8px;">Same entity always written the same way</td></tr>
    <tr><td style="padding: 3px 8px;">Controlled vocabulary</td><td style="padding: 3px 8px;">Fixed list of allowed relations</td></tr>
  </tbody>
</table>

<p style="font-size: 13px; margin: 4px 0;">Goal of this notebook: produce a clean list of triples from all 80 chunks, saved to <code>triples.json</code>, ready to build the knowledge graph.</p>

</div>